In [1]:
import numpy as np
import pandas as pd
import os

# Load the dataset
path_to_load_data = 'c:/Users/Ciro/C/_UGR_PDI/RELIEF-T1D/Datasets/2025-12-05/Preprocessed/diatrend/Final_versions_V4/Filtered_for_prediction/Glucose_measurements_FILTERED_2025-11-20.parquet'
df_data = pd.read_parquet(path_to_load_data)

# Filter valid rows (15min is not NaT)
df_valid_rows = df_data[df_data['15min'].notna()]

# Sort columns by 'Patient_ID' and '15min'
df_valid_rows = df_valid_rows.sort_values(by=['Patient_ID', '15min'])
df_valid_rows.reset_index(drop=True, inplace=True)  # Reset index after filtering

# Group by 'Patient_ID' and convert 'Measurement' to numpy array
patient_dict = {
    pid: group['Measurement'].to_numpy() 
    for pid, group in df_valid_rows.groupby('Patient_ID')
}

date_dict = {
    pid: group['15min'].dt.date.to_numpy() 
    for pid, group in df_valid_rows.groupby('Patient_ID')
}

time_dict = {
    pid: group['15min'].dt.time.to_numpy()
    for pid, group in df_valid_rows.groupby('Patient_ID')
}

print(f"Number of patients with valid data: {len(patient_dict)}")

# Windows creation parameters
history_length = 8  # 120 min
#horizons = [2, 4]  # 2 for 30 min, 4 for 60 min
horizons = [2]
deduplicate_intra_patient = True  # Remove duplicated instances in a patient

path_to_save_windows = 'c:/Users/Ciro/C/_UGR_PDI/RELIEF-T1D/windows/Extra_Fields/'
if not os.path.exists(path_to_save_windows):
    # If it does not exist, create it
    os.makedirs(path_to_save_windows)

Number of patients with valid data: 51


In [2]:
# Extract the time from the timestamp in column 15min
df_valid_rows['15min'].dt.time

0          21:00:00
1          21:15:00
2          21:30:00
3          21:45:00
4          22:00:00
             ...   
2916707    14:30:00
2916708    14:45:00
2916709    15:00:00
2916710    15:15:00
2916711    15:30:00
Name: 15min, Length: 2916712, dtype: object

In [3]:
df_valid_rows['15min'].dt.date

0          2015-12-01
1          2015-12-01
2          2015-12-01
3          2015-12-01
4          2015-12-01
              ...    
2916707    2019-09-27
2916708    2019-09-27
2916709    2019-09-27
2916710    2019-09-27
2916711    2019-09-27
Name: 15min, Length: 2916712, dtype: object

In [4]:
df_valid_rows

,Patient_ID,Measurement_date,Measurement_time,Measurement,15min,5min
0,1,2015-12-01,21:01:07,160.000000,2015-12-01 21:00:00,2015-12-01 21:00:00
1,1,2015-12-01,21:16:08,162.000000,2015-12-01 21:15:00,2015-12-01 21:15:00
2,1,2015-12-01,21:31:13,167.000000,2015-12-01 21:30:00,2015-12-01 21:30:00
3,1,2015-12-01,21:46:07,150.000000,2015-12-01 21:45:00,2015-12-01 21:45:00
4,1,2015-12-01,22:01:07,164.000000,2015-12-01 22:00:00,2015-12-01 22:00:00
...,...,...,...,...,...,...
2916707,53,2019-09-27,14:27:57,167.854619,2019-09-27 14:30:00,2019-09-27 14:30:00
2916708,53,2019-09-27,14:42:57,159.861542,2019-09-27 14:45:00,2019-09-27 14:45:00
2916709,53,2019-09-27,14:57:58,154.865869,2019-09-27 15:00:00,2019-09-27 15:00:00
2916710,53,2019-09-27,15:12:58,150.869330,2019-09-27 15:15:00,2019-09-27 15:15:00


In [5]:
patient_dict[1]

array([160., 162., 167., ..., 152., 116., 124.])

In [6]:
date_dict[1]

array([datetime.date(2015, 12, 1), datetime.date(2015, 12, 1),
       datetime.date(2015, 12, 1), ..., datetime.date(2021, 11, 18),
       datetime.date(2021, 11, 18), datetime.date(2021, 11, 18)],
      dtype=object)

In [7]:
time_dict[1]

array([datetime.time(21, 0), datetime.time(21, 15), datetime.time(21, 30),
       ..., datetime.time(23, 15), datetime.time(23, 30),
       datetime.time(23, 45)], dtype=object)

In [78]:


# Auxiliary function
def get_windows_one_step_walk_forward(bgl_measurement_dict,
                                      history_length,
                                      horizon,
                                      deduplicate_intra_patient) -> pd.DataFrame:
    """
      :param bgl_measurement_dict: Dictionary with patient_id as key and numpy array of temporal series of BGL measurements
      :param history_length: Number of samples used to predict (8 for 120 minutes)
      :param horizon: Prediction horizon in number of windows (2 for 30 minutes, 4 for 60 minutes, and so on)
      :return: Get windows without missing values from one step sliding windows
    """
    df_all_patient_set = pd.DataFrame()
    
    for patient_id, patient_series in bgl_measurement_dict.items():
        # print(f"Clave: {patient_id} — Valor: {patient_series}")

        # Creating the one step sliding window
        x = np.lib.stride_tricks.sliding_window_view(patient_series[:-horizon], history_length)
        x_date = np.lib.stride_tricks.sliding_window_view(date_dict[patient_id][:-horizon], history_length)
        x_time = np.lib.stride_tricks.sliding_window_view(time_dict[patient_id][:-horizon], history_length)
                
        y = np.lib.stride_tricks.sliding_window_view(patient_series[history_length:], horizon)
        # y_date = np.lib.stride_tricks.sliding_window_view(date_dict[patient_id][history_length:], horizon)
        # y_time = np.lib.stride_tricks.sliding_window_view(time_dict[patient_id][history_length:], horizon)

        # Removing rows with missing values (NaN)
        nan_rows_x = np.isnan(x).any(axis=1)  # In input values
        nan_rows_y = np.isnan(y).any(axis=1)  # In output values
        x = x[~(nan_rows_x | nan_rows_y)]  # Remove rows with NaN in either x or y from x
        x_date = x_date[~(nan_rows_x | nan_rows_y)]  # Remove rows with NaN from x_date
        x_time = x_time[~(nan_rows_x | nan_rows_y)]  # Remove rows with NaN from x_time
        
        y = y[~(nan_rows_x | nan_rows_y)]  # Remove rows with NaN in either x or y from y
        # y_date = y_date[~(nan_rows_x | nan_rows_y)]  # Remove rows with NaN in either x or y from y_date
        # y_time = y_time[~(nan_rows_x | nan_rows_y)]  # Remove rows with NaN in either x or y from y_time
        
        # Create DataFrame for the current patient
        df_x = pd.DataFrame(x)  # Input windows
        # df_x = df_x.astype(int)  # Convert to integer type
        # Round to 1 decimal place to avoid floating point issues when comparing for duplicates. This is especially important if the original measurements have decimal values, as it can help to reduce the number of unique values and increase the chances of finding duplicates.
        df_x = df_x.round(1)
        df_x_date = pd.DataFrame(x_date[:, -1])  # Input windows dates
        df_x_time = pd.DataFrame(x_time[:, -1])  # Input windows times
        
        df_y = pd.DataFrame(y[:, -1])  # Output windows (last column of y)
        # df_y = df_y.astype(int)  # Convert to integer type
        # Round to 1 decimal place to avoid floating point issues when comparing for duplicates. This is especially important if the original measurements have decimal values, as it can help to reduce the number of unique values and increase the chances of finding duplicates.
        df_y = df_y.round(1)
        # df_y_date = pd.DataFrame(y_date[:, -1])  # Output windows dates (last column of y_date)
        # df_y_time = pd.DataFrame(y_time[:, -1])  # Output windows times (last column of y_time)
        
        # df_set = pd.concat([df_x, df df_y], axis=1)  # Combine inputs and outputs
        
        # Concatenate all the DataFrames horizontally (inputs, input dates, input times, output, output dates, output times)
        df_set = pd.concat([df_x, df_y, df_x_date, df_x_time], axis=1)        
        # df_set = pd.concat([df_x, df_y, df_x_date, df_x_time, df_y_date, df_y_time], axis=1)        
        # df_set = df_set.astype(int)  # Convert to integer type

        # Renaming columns and adding patient_id.
        # Input: x0 a x7, Output: y
        # new_cols = [f"x{i}" for i in range(8)] + ["y"]  + ["y_date", "y_time"]
        new_cols = [f"x{i}" for i in range(8)] + ["y"] + [f"x_date_{x.shape[1]-1}"] + [f"x_time_{x.shape[1]-1}"]
        df_set.columns = new_cols
        df_set['patient_id'] = patient_id

        # Concat the current patient's DataFrame to the main DataFrame
        df_all_patient_set = pd.concat([df_all_patient_set, df_set], ignore_index=True)

    if deduplicate_intra_patient:
        # Drop duplicated instances in a patient. All columns are considered for duplication, therefore, if a patient has the same input and output values, it will be considered a duplicate, but if there are two equal instances but from different patients, they will not be considered duplicates.
        print("Deduplicating intra-patient instances...")
        print(f"Initial DataFrame shape: {df_all_patient_set.shape}")
        
        # Search for duplicated rows within each patient_id group and mark them as duplicates but only on the input and output columns (x0 to x7 and y), ignoring the date and time columns for duplication. This way, if two rows have the same input and output values but different dates or times, they will be considered duplicates.
        df_all_patient_set['is_duplicate'] = df_all_patient_set.duplicated(subset=[f"x{i}" for i in range(8)] + ["y"] + ['patient_id'], keep='first')
        df_all_patient_set = df_all_patient_set[~df_all_patient_set['is_duplicate']]  # Keep only non-duplicate rows
        df_all_patient_set = df_all_patient_set.drop(columns=['is_duplicate'])  # Drop the helper column
        df_all_patient_set.reset_index(drop=True, inplace=True)  # Reset index after dropping duplicates
        print(f"DataFrame shape after intra-patient deduplication: {df_all_patient_set.shape}")
        
        # df_all_patient_set = df_all_patient_set.drop_duplicates()
        # df_all_patient_set.reset_index(drop=True, inplace=True)
        # print(f"Deduplicated DataFrame shape: {df_all_patient_set.shape}")
    
    return df_all_patient_set


for current_horizon in horizons:
    print(f"Processing horizon: {current_horizon}")
    df_windows = get_windows_one_step_walk_forward(bgl_measurement_dict=patient_dict,
                                                   history_length=history_length,
                                                   horizon=current_horizon,
                                                   deduplicate_intra_patient=deduplicate_intra_patient)

    # NOTE: Replace with your filepath
    df_windows.to_parquet(f'{path_to_save_windows}/windows_horizon_{current_horizon}.parquet')
    print(f"Saved windows for horizon {current_horizon} to parquet files.")
    print("--------------------------------------------------")
    print()


Processing horizon: 2
Deduplicating intra-patient instances...
Initial DataFrame shape: (2333967, 12)
DataFrame shape after intra-patient deduplication: (2322824, 12)
Saved windows for horizon 2 to parquet files.
--------------------------------------------------


In [79]:
df_windows

,x0,x1,x2,x3,x4,x5,x6,x7,y,x_date_7,x_time_7,patient_id
0,160.0,162.0,167.0,150.0,164.0,147.0,130.0,137.0,143.0,2015-12-01,22:45:00,1
1,162.0,167.0,150.0,164.0,147.0,130.0,137.0,131.0,135.0,2015-12-01,23:00:00,1
2,167.0,150.0,164.0,147.0,130.0,137.0,131.0,143.0,143.0,2015-12-01,23:15:00,1
3,177.0,176.0,195.0,194.0,153.0,117.0,100.0,94.0,74.0,2015-12-04,20:45:00,1
4,176.0,195.0,194.0,153.0,117.0,100.0,94.0,83.0,81.0,2015-12-04,21:00:00,1
...,...,...,...,...,...,...,...,...,...,...,...,...
2322819,178.8,178.8,169.9,165.9,160.9,188.8,193.8,191.8,167.9,2019-09-27,14:00:00,53
2322820,178.8,169.9,165.9,160.9,188.8,193.8,191.8,185.8,159.9,2019-09-27,14:15:00,53
2322821,169.9,165.9,160.9,188.8,193.8,191.8,185.8,167.9,154.9,2019-09-27,14:30:00,53
2322822,165.9,160.9,188.8,193.8,191.8,185.8,167.9,159.9,150.9,2019-09-27,14:45:00,53


In [81]:
df_old_windows = pd.read_parquet('c:/Users/Ciro/C/_UGR_PDI/RELIEF-T1D/Datasets/2025-12-05/Preprocessed/diatrend/files_for_crossvalidation/windows/windows_horizon_2.parquet')
df_old_windows

,x0,x1,x2,x3,x4,x5,x6,x7,y,patient_id
0,160,162,167,150,164,147,130,137,143,1
1,162,167,150,164,147,130,137,131,135,1
2,167,150,164,147,130,137,131,143,143,1
3,177,176,195,194,153,117,100,94,74,1
4,176,195,194,153,117,100,94,83,81,1
...,...,...,...,...,...,...,...,...,...,...
2322819,178,178,169,165,160,188,193,191,167,53
2322820,178,169,165,160,188,193,191,185,159,53
2322821,169,165,160,188,193,191,185,167,154,53
2322822,165,160,188,193,191,185,167,159,150,53


In [48]:
df_old_windows.shape[0]

2322824